# The Job Requirements Bubble Up
Created by James Davis (https://www.linkedin.com/in/jamesdavismakes)

### Week 1 Challenge
> Come up with some kind of challenge that involves distilling some information, synthesizing it, and producting some output.

### What is the *Job Requirements Bubble Up*?
The plan (which I'm doing live, building the car while driving it) is to read a number of job pages and "bubbling up" the most common term that come up.

For instance, out of let's say 10 software engineer job pages, if JSON is mentioned a lot then it will be higher in an prioritized list of terms aspiring engineers should focus on.

How are we going to do this? Welp, let's find out!

### Setup

First, let's import all of the necessary libraries. Also, let's not reinvent the wheel and just ~~steal~~ lovingly borrow Ed's scrapping functions.

In [39]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

I'm using Google Gemini so I'll setup my environment to reflect that.

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = 'gemini-3.5-flash'
MODEL_CHEAP = 'gemini-3.1-flash-lite'

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and (api_key.startswith('AIz') or api_key.startswith('AQ.')):
    print("API key looks good!")
else:
    print("Your API key doesn't start with AIz or AQ. Please check your .env file and try again.")

client = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

So far so good! I'm including both 3.5 Flash and 3.1 Flash Lite just as a way to easily switch between the two and evaluate the responses. I don't intend to use both.

### Step 1: Fetch job links
Honestly, I think this will be the hard part.

Using Ed's website fetching function in *scraper.py*, I'm going to try pulling a job post from websites like LinkedIn and Indeed.

Spoilers: LinkedIn didn't parse well, and Indeed requires javascript. Most job sites are not cool with what we're doing, so I went with the site *WeWorkRemotely*.

First, let's get the links off of the search.

In [ ]:
job_role = "AI Engineer"
contents = fetch_website_links(f"https://weworkremotely.com/remote-jobs/search?term={job_role}")
print(contents)

And let's use AI to decide which links are actually job links.

In [34]:
prompt_system_links = f"""
You are provided with a list of links found on a job search website.
You are highly capable in deciding which links lead to job descriptions.
You should respond in JSON as an array of strings.
"""

In [48]:
def get_links_user_prompt(role):
    prompt_user = f"""
Here is a list of links from the job website https://weworkremotely.com for the role of {role}.
Please decide which of these links are relevant in terms of linking to a job description.
Please exclude relative links.
Please respond with the full https URL.

Links:
    """

    links = fetch_website_links(f"https://weworkremotely.com/remote-jobs/search?term={role}")
    prompt_user += "\n".join(links)
    return prompt_user

In [43]:
def ai_select_relevant_links(role):
    response = client.chat.completions.create(
        model=MODEL_CHEAP,
        messages = [
            { "role" : "system", "content" : prompt_system_links },
            { "role" : "user", "content" : get_links_user_prompt(role) }
        ],
        response_format = { "type" : "json_object" }
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

Alright, let's see how Flash Lite does.

In [44]:
ROLE = "AI Engineer"

In [ ]:
ai_select_relevant_links(ROLE)

Alright! We have a list of actually relevant links. They all link to job descriptions. Hard part done.

### Step 2: Let's make some bubbles!

So, to solidify what the hell a bubble is, a bubble is essentially a skill you'd find on a resume or job description. i.e. "I'm experienced in **Java**, **Powerpoint**, **Cursor**, etc." And bubbles rise by the number of times they are mentioned.

So first, let's go through all the links from Step 1:

In [56]:
def fetch_job_posts(role):
    relevant_links = ai_select_relevant_links(role)
    result = "## Job Pages:"

    for link in relevant_links:
        result += f"\n\nLink: {link}\n"
        result += fetch_website_contents(link)

    return result
    

In [ ]:
print(fetch_job_posts("Android Engineer"))

In [83]:
prompt_system_bubbles = """
You are a job analyzer with proven experience identifying important skills from a job post.
Using this skill, you are great at identifying how often skills are mentioned.
When considering skills, lean towards being specific. For instance, if choosing between "software developemnt" and "python" choose python.
Respond in markdown without code blocks.
"""

In [84]:
def ai_get_bubble_doc_user_prompt(role):
    prompt_user = f"""
You are helping someone interested in becoming a {role} figure out the skills needed to do that role.
Here are the contents of a job search they made. Using the descriptions, you need to keep count of how often skills are mentioned.
Using this information, build a short document that outlines these stats. Format as follows:

The job title they searched as the title
A short description of the job title, no more than 280 characters
A section that lists (at most) 15 of the top mentioned skills in the job descriptions, sorted in order of how often mentioned -- most mentions on top.
A short description of what the skill involves, 140 characters max, and how often it was mentioned.
A section that lists specific skills you think is important to focus on (at most 5). Assume they are starting fresh.
With a short 140-character max reason why for each.
    """

    prompt_user += "\n\nHere are the job posts:\n"
    prompt_user += fetch_job_posts(role)
    return prompt_user


Alright, the moment of truth...

In [81]:
def create_job_bubble_report(job_role):
    response = client.chat.completions.create(
        model=MODEL_CHEAP,
        messages = [
            { "role" : "system", "content" : prompt_system_bubbles },
            { "role" : "user", "content" : ai_get_bubble_doc_user_prompt(job_role)}
        ]
    )

    result = response.choices[0].message.content
    display(Markdown(result))

*I ended up sticking with Flash Lite because the output from it and Flash Heavy weren't that different in terms of quality.*

In [86]:
create_job_bubble_report("AI Engineer")

# AI Engineer

This role involves designing, building, and maintaining AI-powered systems, including LLM integration, agentic workflows, and infrastructure to support production-scale machine learning applications.

### Top Mentioned Skills

1. **LLMs (Large Language Models)**: Core technology for building conversational AI and agentic systems. (Mentions: 6)
2. **AI Agents**: Autonomous systems designed to execute tasks or workflows. (Mentions: 5)
3. **RAG (Retrieval-Augmented Generation)**: Frameworks to connect LLMs to external data for accuracy. (Mentions: 3)
4. **Python**: Primary programming language for AI/ML development and scripting. (Mentions: 3)
5. **AWS**: Cloud infrastructure provider for deploying AI systems and models. (Mentions: 2)
6. **Infrastructure**: Designing and managing the hardware/software foundations for AI. (Mentions: 2)
7. **Prompt Engineering**: Designing inputs to guide LLM behavior for specific outputs. (Mentions: 2)
8. **Evaluation**: Testing and validating AI model performance and reliability. (Mentions: 2)
9. **Observability**: Monitoring systems to troubleshoot and optimize AI performance. (Mentions: 1)
10. **Automation**: Using code to streamline repetitive tasks and AI workflows. (Mentions: 1)
11. **SQL**: Managing data storage and retrieval in database systems. (Mentions: 1)
12. **System Architecture**: Designing the structure of scalable, reliable software. (Mentions: 1)
13. **Cloud Computing**: Managing remote resources for hosting scalable applications. (Mentions: 1)
14. **Containerization (Docker)**: Packaging software into isolated units for easy deployment. (Mentions: 1)
15. **API Development**: Creating interfaces for software components to communicate. (Mentions: 1)

### Recommended Skills for Beginners

1. **Python**: The universal language of AI; you cannot build modern AI systems without it.
2. **LLM APIs (OpenAI/Anthropic)**: Understanding how to prompt and interact with models is the first step to building apps.
3. **RAG Pipelines**: Learning how to ground AI in actual data is critical for professional use cases.
4. **Cloud Basics (AWS/Azure)**: AI needs scale; knowing how to deploy in the cloud is required for production jobs.
5. **Prompt Engineering**: Mastering how to talk to models is the foundational skill for all AI engineering work.

And there you have it! Also pretty cool that the recommended section outlines the steps I'm actively following now by taking this course haha

There are some layout stuff I want to fix but it's time to jump into Week 2!